# LeetCode 3203 — Find Minimum Diameter After Merging Two Trees bằng Python

- Link: https://leetcode.com/problems/find-minimum-diameter-after-merging-two-trees/
- Độ khó: **Hard**
- Chủ đề: **Đường kính cây** (BFS hai lần), **độ lệch tâm (eccentricity)** và **bán kính** `⌈d/2⌉`; nối hai **tâm** ⇒ đáp án `max(d1, d2, ⌈d1/2⌉ + ⌈d2/2⌉ + 1)`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumDiameterAfterMerge(self, edges1: List[List[int]], edges2: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Tính đường kính cây bằng **BFS hai lần** (từ một đỉnh bất kỳ tới đỉnh xa nhất `a`, rồi từ `a` tới đỉnh xa nhất).
2. Hiểu đường đi dài nhất sau khi nối `u–v` hoặc nằm hẳn trong một cây, hoặc đi qua cạnh mới: `ecc1(u) + 1 + ecc2(v)`.
3. Độ lệch tâm nhỏ nhất (bán kính) của cây đường kính `d` là `⌈d/2⌉`, đạt tại **tâm** (trung điểm đường kính).
4. Kết hợp: `max(d1, d2, ⌈d1/2⌉ + ⌈d2/2⌉ + 1)`.
5. Kiểm chứng bằng brute thử mọi cặp nối.

## 2. Đề bài & ví dụ

Có hai cây vô hướng `n` và `m` đỉnh (đánh số `0..n−1`, `0..m−1`) cho bởi `edges1`, `edges2`. Phải nối **một** đỉnh của cây 1 với **một** đỉnh của cây 2 bằng một cạnh. Trả về **đường kính nhỏ nhất** có thể của cây kết quả (đường kính = số cạnh của đường đi dài nhất).

**Ví dụ 1:** `edges1 = [[0,1],[0,2],[0,3]]`, `edges2 = [[0,1]]` → `3`.

**Ví dụ 2:** `edges1 = edges2 = [[0,1],[0,2],[0,3],[2,4],[2,5],[3,6],[2,7]]` → `5`.

**Ràng buộc:** `1 ≤ n, m ≤ 10^5`; `edges1.length = n − 1`, `edges2.length = m − 1`; đầu vào là cây hợp lệ (có thể **rỗng** khi chỉ có 1 đỉnh).

In [1]:
from typing import List
from collections import deque
import random, time

examples = [([[0, 1], [0, 2], [0, 3]], [[0, 1]], 3),
            ([[0, 1], [0, 2], [0, 3], [2, 4], [2, 5], [3, 6], [2, 7]],
             [[0, 1], [0, 2], [0, 3], [2, 4], [2, 5], [3, 6], [2, 7]], 5)]
for e1, e2, exp in examples:
    print(len(e1) + 1, 'đỉnh và', len(e2) + 1, 'đỉnh -> kỳ vọng', exp)

4 đỉnh và 2 đỉnh -> kỳ vọng 3
8 đỉnh và 8 đỉnh -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. BFS hai lần tìm đường kính

Từ đỉnh bất kỳ `s`, đỉnh xa nhất `a` luôn là **một đầu** của một đường kính; BFS từ `a` cho khoảng cách xa nhất = đường kính.

In [2]:
def build(edges):
    n = len(edges) + 1
    g = [[] for _ in range(n)]
    for a, b in edges:
        g[a].append(b); g[b].append(a)
    return g

def bfs(g, s):
    dist = [-1] * len(g); dist[s] = 0; q = deque([s])
    while q:
        u = q.popleft()
        for v in g[u]:
            if dist[v] < 0:
                dist[v] = dist[u] + 1; q.append(v)
    return dist

def diameter(edges):
    g = build(edges)
    d0 = bfs(g, 0); a = max(range(len(g)), key=d0.__getitem__)
    da = bfs(g, a)
    return max(da)

for e1, e2, _ in examples:
    print('đường kính:', diameter(e1), diameter(e2))

đường kính: 2 1
đường kính: 4 4


### 3.2. Độ lệch tâm, bán kính, tâm

`ecc(u) = max_v dist(u, v)`. Với cây đường kính `d`: `min_u ecc(u) = ⌈d/2⌉` (đạt tại đỉnh giữa đường kính; nếu `d` lẻ có hai tâm kề nhau). Lý do: mọi đỉnh `u` phải đi tới hai đầu `a, b` của đường kính, `dist(u, a) + dist(u, b) ≥ d` ⇒ `ecc(u) ≥ ⌈d/2⌉`; và đỉnh giữa đạt dấu bằng (nếu có đỉnh xa hơn thì đường kính dài hơn `d`).

In [3]:
def eccentricities(edges):
    g = build(edges)
    return [max(bfs(g, s)) for s in range(len(g))]

for e1, e2, _ in examples:
    for e in (e1, e2):
        ec = eccentricities(e); d = diameter(e)
        print('ecc =', ec, '| min =', min(ec), '| ⌈d/2⌉ =', (d + 1) // 2)

ecc = [1, 2, 2, 2] | min = 1 | ⌈d/2⌉ = 1
ecc = [1, 1] | min = 1 | ⌈d/2⌉ = 1
ecc = [2, 3, 3, 3, 4, 4, 4, 4] | min = 2 | ⌈d/2⌉ = 2
ecc = [2, 3, 3, 3, 4, 4, 4, 4] | min = 2 | ⌈d/2⌉ = 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Sau khi nối `u` (cây 1) với `v` (cây 2), một đường đi dài nhất:

- nằm trọn trong cây 1 ⇒ `≤ d1`, trong cây 2 ⇒ `≤ d2` (không đổi khi nối);
- hoặc đi qua cạnh mới ⇒ dài nhất là `ecc1(u) + 1 + ecc2(v)`.

Muốn nhỏ nhất ⇒ chọn `u`, `v` có độ lệch tâm nhỏ nhất (tâm) ⇒ `⌈d1/2⌉ + 1 + ⌈d2/2⌉`. Ví dụ 1: `d1 = 2`, `d2 = 1` ⇒ `max(2, 1, 1 + 1 + 1) = 3`.

## 5. Cách 1 — Brute Force: thử mọi cặp `(u, v)`, tính đường kính cây ghép

In [4]:
def brute(edges1, edges2):
    n = len(edges1) + 1; m = len(edges2) + 1
    base = [list(e) for e in edges1] + [[a + n, b + n] for a, b in edges2]
    best = None
    for u in range(n):
        for v in range(m):
            d = diameter(base + [[u, v + n]])
            best = d if best is None else min(best, d)
    return best

for e1, e2, exp in examples:
    print('brute:', brute(e1, e2), '| kỳ vọng', exp)

brute: 3 | kỳ vọng 3
brute: 5 | kỳ vọng 5


**Độ phức tạp:** `O(n · m · (n + m))` — chỉ cho cây nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

```
d1 = diameter(edges1); d2 = diameter(edges2)
return max(d1, d2, (d1 + 1) // 2 + (d2 + 1) // 2 + 1)
```

Mỗi đường kính `O(n)` bằng BFS hai lần.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Độ lệch tâm của mọi đỉnh `O(n² + m²)`

In [5]:
def by_eccentricity(edges1, edges2):
    e1 = eccentricities(edges1); e2 = eccentricities(edges2)
    return max(max(e1), max(e2), min(e1) + min(e2) + 1)     # max(ecc) = đường kính

for e1, e2, exp in examples:
    print('ecc:', by_eccentricity(e1, e2), '| kỳ vọng', exp)

ecc: 3 | kỳ vọng 3
ecc: 5 | kỳ vọng 5


### 7.2. Cách 3 — BFS hai lần + công thức (lời giải chính)

In [6]:
def two_bfs_formula(edges1, edges2):
    d1 = diameter(edges1); d2 = diameter(edges2)
    return max(d1, d2, (d1 + 1) // 2 + (d2 + 1) // 2 + 1)

for e1, e2, exp in examples:
    print('2 BFS:', two_bfs_formula(e1, e2), '| kỳ vọng', exp)

2 BFS: 3 | kỳ vọng 3
2 BFS: 5 | kỳ vọng 5


### 7.3. Cách 4 — Bóc lá theo tầng (tìm tâm như LC 310)

Lặp bỏ toàn bộ lá; sau `L` vòng còn lại 1 tâm (đường kính `2L`) hoặc 2 tâm kề nhau (đường kính `2L + 1`).

In [7]:
def diameter_by_peeling(edges):
    n = len(edges) + 1
    if n == 1:
        return 0
    g = build(edges); deg = [len(x) for x in g]
    layer = [u for u in range(n) if deg[u] == 1]; remaining = n; L = 0
    while remaining > 2:
        remaining -= len(layer); nxt = []
        for u in layer:
            for v in g[u]:
                deg[v] -= 1
                if deg[v] == 1:
                    nxt.append(v)
        layer = nxt; L += 1
    return 2 * L + (1 if remaining == 2 else 0)

def peeling_formula(edges1, edges2):
    d1 = diameter_by_peeling(edges1); d2 = diameter_by_peeling(edges2)
    return max(d1, d2, (d1 + 1) // 2 + (d2 + 1) // 2 + 1)

for e1, e2, exp in examples:
    print('bóc lá:', peeling_formula(e1, e2), '| kỳ vọng', exp)

bóc lá: 3 | kỳ vọng 3
bóc lá: 5 | kỳ vọng 5


### 7.4. Cách 5 — DP cây (chiều cao hai nhánh sâu nhất), duyệt lặp

In [8]:
def diameter_by_dp(edges):
    g = build(edges); n = len(g)
    parent = [-1] * n; order = []; stack = [0]; seen = [False] * n; seen[0] = True
    while stack:
        u = stack.pop(); order.append(u)
        for v in g[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; stack.append(v)
    h = [0] * n; best = 0
    for u in reversed(order):                       # con trước cha
        a = b = 0                                   # hai chiều cao con lớn nhất (+1)
        for v in g[u]:
            if v != parent[u]:
                x = h[v] + 1
                if x > a:
                    a, b = x, a
                elif x > b:
                    b = x
        h[u] = a; best = max(best, a + b)
    return best

def dp_formula(edges1, edges2):
    d1 = diameter_by_dp(edges1); d2 = diameter_by_dp(edges2)
    return max(d1, d2, (d1 + 1) // 2 + (d2 + 1) // 2 + 1)

for e1, e2, exp in examples:
    print('DP cây:', dp_formula(e1, e2), '| kỳ vọng', exp)

DP cây: 3 | kỳ vọng 3
DP cây: 5 | kỳ vọng 5


## 8. TRACE từng bước

Trace BFS hai lần trên cây 1 của ví dụ 2, rồi áp công thức.

In [9]:
e1 = examples[1][0]; g = build(e1)
d0 = bfs(g, 0); a = max(range(len(g)), key=d0.__getitem__)
print('BFS từ 0: dist =', d0, '-> đỉnh xa nhất a =', a)
da = bfs(g, a); b = max(range(len(g)), key=da.__getitem__)
print(f'BFS từ {a}: dist =', da, f'-> đỉnh xa nhất b = {b}, đường kính d1 = {da[b]}')
# khôi phục đường kính và tâm
par = {a: None}; q = deque([a])
while q:
    u = q.popleft()
    for v in g[u]:
        if v not in par:
            par[v] = u; q.append(v)
path = [b]
while par[path[-1]] is not None:
    path.append(par[path[-1]])
print('đường kính:', path, '| tâm:', path[len(path) // 2] if len(path) % 2 else path[len(path) // 2 - 1: len(path) // 2 + 1])
d1 = d2 = da[b]
print(f'công thức: max({d1}, {d2}, {(d1 + 1) // 2} + {(d2 + 1) // 2} + 1) =', max(d1, d2, (d1 + 1) // 2 + (d2 + 1) // 2 + 1))

BFS từ 0: dist = [0, 1, 1, 1, 2, 2, 2, 2] -> đỉnh xa nhất a = 4
BFS từ 4: dist = [2, 3, 1, 3, 0, 2, 4, 2] -> đỉnh xa nhất b = 6, đường kính d1 = 4
đường kính: [6, 3, 0, 2, 4] | tâm: 0
công thức: max(4, 4, 2 + 2 + 1) = 5


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def minimumDiameterAfterMerge(self, edges1: List[List[int]], edges2: List[List[int]]) -> int:
        def diam(edges):
            n = len(edges) + 1
            g = [[] for _ in range(n)]
            for a, b in edges:
                g[a].append(b)
                g[b].append(a)

            def far(s):
                dist = [-1] * n
                dist[s] = 0
                q = deque([s])
                last = s
                while q:
                    u = q.popleft()
                    last = u
                    for v in g[u]:
                        if dist[v] < 0:
                            dist[v] = dist[u] + 1
                            q.append(v)
                return last, dist[last]

            a, _ = far(0)
            _, d = far(a)
            return d

        d1 = diam(edges1)
        d2 = diam(edges2)
        return max(d1, d2, (d1 + 1) // 2 + (d2 + 1) // 2 + 1)

for e1, e2, exp in examples:
    got = Solution().minimumDiameterAfterMerge(e1, e2)
    print(got, 'OK' if got == exp else 'SAI')

3 OK
5 OK


(Đỉnh lấy ra **cuối cùng** khỏi hàng đợi BFS có khoảng cách lớn nhất — không cần `max` riêng.)

## 10. Vì sao thuật toán đúng

1. **Phân loại đường đi** trong cây ghép (nối `u–v`): đường không dùng cạnh mới nằm trong một cây ⇒ dài nhất `max(d1, d2)` (không phụ thuộc `u, v`). Đường dùng cạnh mới có dạng `x ⇝ u – v ⇝ y` ⇒ dài nhất `ecc1(u) + 1 + ecc2(v)`.
2. Đường kính cây ghép `= max(d1, d2, ecc1(u) + 1 + ecc2(v))`; chỉ số hạng cuối phụ thuộc lựa chọn, và nó đạt min khi `ecc1(u) = r1`, `ecc2(v) = r2` (bán kính).
3. **Bán kính `= ⌈d/2⌉`** (mục 3.2) ⇒ công thức. Đã kiểm `min(ecc) = ⌈d/2⌉` và đối chiếu brute ở mục 12.
4. **BFS hai lần:** đỉnh xa nhất từ một đỉnh bất kỳ là đầu mút của một đường kính (tính chất chuẩn của cây, chứng minh bằng phản chứng với đường kính).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(n m (n + m))` | `O(n + m)` |
| 2. Mọi độ lệch tâm | `O(n² + m²)` | `O(n + m)` |
| 3. BFS hai lần | `O(n + m)` | `O(n + m)` |
| 4. Bóc lá | `O(n + m)` | `O(n + m)` |
| 5. DP cây lặp | `O(n + m)` | `O(n + m)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'ecc': by_eccentricity, '2bfs': two_bfs_formula, 'peel': peeling_formula, 'dp': dp_formula,
         'submit': lambda a, b: Solution().minimumDiameterAfterMerge(a, b)}
path = lambda k: [[i, i + 1] for i in range(k - 1)]
edge = [([], [], 1), ([], [[0, 1]], 2), (path(6), [], 5), (path(3), path(3), 3),
        (path(4), path(4), 5), ([[0, 1], [0, 2], [0, 3], [0, 4]], [], 2)]
for e1, e2, exp in examples + edge:
    assert brute(e1, e2) == exp, (e1, e2, brute(e1, e2))
    for name, f in impls.items():
        assert f(e1, e2) == exp, (name, e1, e2)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: hai cây một đỉnh ⇒ chỉ có cạnh mới ⇒ `1`. Đường 6 đỉnh (`d = 5`) nối với 1 đỉnh: `⌈5/2⌉ + 0 + 1 = 4 < 5` ⇒ `max` giữ `5`. Hai đường 4 đỉnh (`d = 3`): `2 + 2 + 1 = 5`.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [12]:
rng = random.Random(3203)
def rand_tree(n):
    return [[i, rng.randint(0, i - 1)] for i in range(1, n)]

cnt = 0
for _ in range(250):
    e1 = rand_tree(rng.randint(1, 7)); e2 = rand_tree(rng.randint(1, 7))
    exp = brute(e1, e2)
    for name, f in impls.items():
        assert f(e1, e2) == exp, (name, e1, e2)
    cnt += 1
print('Khớp brute trên', cnt, 'cặp cây ngẫu nhiên.')

Khớp brute trên 250 cặp cây ngẫu nhiên.


### 12.3. Ca lớn `n = m = 10^5`

In [13]:
N = 10 ** 5
big = {
    'đường thẳng': (path(N), path(N)),
    'ngẫu nhiên': (rand_tree(N), rand_tree(N)),
    'sao + đường': ([[0, i] for i in range(1, N)], path(N)),
}
for name, (e1, e2) in big.items():
    t = time.time(); x = Solution().minimumDiameterAfterMerge(e1, e2); t1 = time.time() - t
    y = peeling_formula(e1, e2); z = dp_formula(e1, e2)
    assert x == y == z
    print(f'{name:12s}: {x:7d} | nộp {t1:.2f}s')
assert Solution().minimumDiameterAfterMerge(path(N), path(N)) == 2 * ((N - 1 + 1) // 2) + 1
print('Hai đường N đỉnh: 2·⌈(N−1)/2⌉ + 1: đúng.')

đường thẳng :  100001 | nộp 0.22s


ngẫu nhiên  :      51 | nộp 0.49s


sao + đường :   99999 | nộp 0.10s


Hai đường N đỉnh: 2·⌈(N−1)/2⌉ + 1: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `⌊d/2⌋` thay vì `⌈d/2⌉`

In [14]:
def wrong_floor(e1, e2):
    d1 = diameter(e1); d2 = diameter(e2)
    return max(d1, d2, d1 // 2 + d2 // 2 + 1)                       # SAI

print('sai :', wrong_floor(*examples[0][:2]), '| đúng:', Solution().minimumDiameterAfterMerge(*examples[0][:2]))

sai : 2 | đúng: 3


**Vì sao sai:** đường kính lẻ (`d = 1` ở cây 2) có tâm ở **giữa cạnh**; mọi đỉnh có độ lệch tâm `≥ ⌈d/2⌉`.

### Lỗi 2 — Quên `max(d1, d2)`

In [15]:
def wrong_no_max(e1, e2):
    d1 = diameter(e1); d2 = diameter(e2)
    return (d1 + 1) // 2 + (d2 + 1) // 2 + 1                         # SAI

print('sai :', wrong_no_max(path(6), []), '| đúng:', Solution().minimumDiameterAfterMerge(path(6), []))

sai : 4 | đúng: 5


**Vì sao sai:** đường kính cũ của một cây không biến mất sau khi nối; khi cây kia nhỏ, nó lớn hơn đường qua cạnh mới.

### Lỗi 3 — Nối hai đỉnh 0 với nhau (hoặc đỉnh bất kỳ)

In [16]:
def wrong_connect_zero(e1, e2):
    n = len(e1) + 1
    return diameter([list(e) for e in e1] + [[a + n, b + n] for a, b in e2] + [[0, n]])   # SAI

print('sai :', wrong_connect_zero(path(3), path(3)), '| đúng:', Solution().minimumDiameterAfterMerge(path(3), path(3)))

sai : 5 | đúng: 3


**Vì sao sai:** đỉnh 0 của đường `0–1–2` là **lá** (độ lệch tâm 2), không phải tâm. Nối hai lá ⇒ `2 + 1 + 2 = 5`; nối hai tâm ⇒ `3`.

### Lỗi 4 — Đường kính = khoảng cách xa nhất **từ đỉnh 0** (chỉ BFS một lần)

In [17]:
def wrong_one_bfs(e1, e2):
    d1 = max(bfs(build(e1), 0)); d2 = max(bfs(build(e2), 0))         # SAI: đó là ecc(0), không phải đường kính
    return max(d1, d2, (d1 + 1) // 2 + (d2 + 1) // 2 + 1)

t = [[0, 1], [0, 2], [2, 3]]
print('sai :', wrong_one_bfs(t, t), '| đúng:', Solution().minimumDiameterAfterMerge(t, t))

sai : 3 | đúng: 5


**Vì sao sai:** trong cây `1–0–2–3`, `ecc(0) = 2` nhưng đường kính `1–0–2–3` dài `3`. Phải BFS lần hai từ đỉnh xa nhất.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Nối/ghép cây, tối thiểu đường kính/chiều cao" ⇒ nghĩ đến **tâm** và **bán kính** `⌈d/2⌉`.
- Đường kính cây: BFS hai lần (hoặc DP cây) — tránh đệ quy sâu trong Python với `n = 10^5`.
- Cây một đỉnh: `edges` rỗng — nhớ `n = len(edges) + 1`.

## 15. Biến thể của bài

1. **LC 310 Minimum Height Trees:** tìm các tâm (bóc lá).
2. **LC 1245 Tree Diameter**, **LC 543** (cây nhị phân).
3. Ghép **k** cây bằng `k − 1` cạnh ⇒ `max(max d_i, r1 + r2 + 1, r2 + r3 + 2)` với `r1 ≥ r2 ≥ r3` là các bán kính lớn nhất (bài tập 4).
4. Đếm số cặp `(u, v)` cho đường kính nhỏ nhất (bài tập 5).

In [18]:
# Biến thể 1 (LC 310): tâm của cây = các đỉnh có độ lệch tâm nhỏ nhất
def centers(edges):
    n = len(edges) + 1
    if n == 1:
        return [0]
    g = build(edges); deg = [len(x) for x in g]
    layer = [u for u in range(n) if deg[u] == 1]; remaining = n
    while remaining > 2:
        remaining -= len(layer); nxt = []
        for u in layer:
            for v in g[u]:
                deg[v] -= 1
                if deg[v] == 1:
                    nxt.append(v)
        layer = nxt
    return sorted(layer)

for _ in range(200):
    e = rand_tree(rng.randint(1, 9)); ec = eccentricities(e)
    assert centers(e) == [u for u in range(len(ec)) if ec[u] == min(ec)]
print('Biến thể LC 310: tâm (bóc lá) = các đỉnh có độ lệch tâm nhỏ nhất.')

Biến thể LC 310: tâm (bóc lá) = các đỉnh có độ lệch tâm nhỏ nhất.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi cặp nối | `O(nm(n+m))` | Oracle |
| 2 | Mọi độ lệch tâm | `O(n² + m²)` | Trực tiếp theo lập luận |
| 3 | BFS hai lần + công thức | `O(n + m)` | **Nộp bài** |
| 4 | Bóc lá | `O(n + m)` | Ra luôn tâm |
| 5 | DP cây lặp | `O(n + m)` | Không cần BFS |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `edges1 = [[0,1]]`, `edges2 = [[0,1]]`.

**Bài 2 (tính tay):** cây 1 là đường 5 đỉnh, cây 2 là đường 4 đỉnh.

**Bài 3 (điền TODO):** trả về một cặp `(u, v)` đạt đường kính nhỏ nhất.

**Bài 4 (tự viết lại):** ghép **ba** cây bằng hai cạnh, tối thiểu đường kính.

**Bài 5 (mini challenge):** đếm số cặp `(u, v)` mà nối `u–v` cho đường kính nhỏ nhất, `O(n log n + m log m)`.

In [19]:
def best_pair_todo(edges1, edges2):
    # TODO
    raise NotImplementedError

def merge_three_todo(e1, e2, e3):
    # TODO
    raise NotImplementedError

def count_best_pairs_todo(edges1, edges2):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `d1 = d2 = 1`. (2) `⌈1/2⌉ = 1`. (3) `max(1, 1, 3) = 3`.

**Bài 2.** (1) `d1 = 4`, `d2 = 3`. (2) Bán kính `2` và `2`. (3) `max(4, 3, 5) = 5`.

**Bài 3.** (1) Lấy một tâm của mỗi cây. (2) Tâm = đỉnh giữa đường kính (khôi phục bằng mảng cha khi BFS lần hai). (3) Kiểm bằng `diameter` của cây ghép.

**Bài 4.** (1) Có một cây "giữa" `B` nối với cả `A` và `C`. (2) Đường đi `A ⇝ B ⇝ C` dài `rA + 1 + (khoảng cách hai điểm nối trong B) + 1 + rC`; nối cả hai vào **tâm** của `B`. (3) Thử mọi cách chọn cây giữa: `max(dA, dB, dC, rA + rB + 1, rC + rB + 1, rA + rC + 2)`, lấy min — tương đương sắp bán kính giảm dần `r1 ≥ r2 ≥ r3` và đáp án `max(max d, r1 + r2 + 1, r2 + r3 + 2)`.

**Bài 5.** (1) `D* = max(d1, d2, r1 + r2 + 1)`. (2) Cặp tốt ⇔ `ecc1(u) + ecc2(v) + 1 ≤ D*`. (3) Sắp `ecc2`, với mỗi `u` dùng `bisect` đếm `v`. Độ lệch tâm mọi đỉnh tính `O(n)` bằng `ecc(u) = max(dist(u, a), dist(u, b))` với `a, b` hai đầu đường kính.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().minimumDiameterAfterMerge([[0, 1]], [[0, 1]]) == 3
assert Solution().minimumDiameterAfterMerge(path(5), path(4)) == 5

def a_center(edges):
    g = build(edges); n = len(g)
    d0 = bfs(g, 0); a = max(range(n), key=d0.__getitem__)
    par = [-1] * n; dist = [-1] * n; dist[a] = 0; q = deque([a])
    while q:
        u = q.popleft()
        for v in g[u]:
            if dist[v] < 0:
                dist[v] = dist[u] + 1; par[v] = u; q.append(v)
    b = max(range(n), key=dist.__getitem__)
    p = [b]
    while p[-1] != a:
        p.append(par[p[-1]])
    return p[len(p) // 2]

def best_pair(edges1, edges2):
    return a_center(edges1), a_center(edges2)

for _ in range(200):
    e1 = rand_tree(rng.randint(1, 8)); e2 = rand_tree(rng.randint(1, 8))
    u, v = best_pair(e1, e2); n = len(e1) + 1
    merged = [list(e) for e in e1] + [[a + n, b + n] for a, b in e2] + [[u, v + n]]
    assert diameter(merged) == Solution().minimumDiameterAfterMerge(e1, e2)
print('Bài 1: 3 | Bài 2: 5 | Bài 3: nối hai tâm đạt tối ưu. Ví dụ 2:', best_pair(*examples[1][:2]))

Bài 1: 3 | Bài 2: 5 | Bài 3: nối hai tâm đạt tối ưu. Ví dụ 2: (0, 0)


In [21]:
# Bài 4, 5
from bisect import bisect_right
import itertools

def merge_three(e1, e2, e3):
    ds = [diameter(e) for e in (e1, e2, e3)]
    rs = sorted(((d + 1) // 2 for d in ds), reverse=True)
    return max(max(ds), rs[0] + rs[1] + 1, rs[1] + rs[2] + 2)

def brute_three(e1, e2, e3):
    trees = [e1, e2, e3]; best = None
    for mid in range(3):
        A, C = [t for i, t in enumerate(trees) if i != mid]; B = trees[mid]
        na, nb_, nc = len(A) + 1, len(B) + 1, len(C) + 1
        base = ([list(e) for e in A] + [[x + na, y + na] for x, y in B] +
                [[x + na + nb_, y + na + nb_] for x, y in C])
        for u, v, w, z in itertools.product(range(na), range(nb_), range(nc), range(nb_)):
            d = diameter(base + [[u, v + na], [w + na + nb_, z + na]])
            best = d if best is None else min(best, d)
    return best

def all_ecc(edges):
    g = build(edges); n = len(g)
    d0 = bfs(g, 0); a = max(range(n), key=d0.__getitem__)
    da = bfs(g, a); b = max(range(n), key=da.__getitem__); db = bfs(g, b)
    return [max(x, y) for x, y in zip(da, db)]

def count_best_pairs(edges1, edges2):
    e1 = all_ecc(edges1); e2 = sorted(all_ecc(edges2))
    D = Solution().minimumDiameterAfterMerge(edges1, edges2)
    return sum(bisect_right(e2, D - 1 - x) for x in e1)

for _ in range(60):
    ts = [rand_tree(rng.randint(1, 4)) for _ in range(3)]
    assert merge_three(*ts) == brute_three(*ts), ts
for _ in range(150):
    e1 = rand_tree(rng.randint(1, 7)); e2 = rand_tree(rng.randint(1, 7)); n = len(e1) + 1
    base = [list(e) for e in e1] + [[a + n, b + n] for a, b in e2]
    D = Solution().minimumDiameterAfterMerge(e1, e2)
    exp = sum(1 for u in range(n) for v in range(len(e2) + 1) if diameter(base + [[u, v + n]]) == D)
    assert count_best_pairs(e1, e2) == exp
    assert all_ecc(e1) == eccentricities(e1)
print('Bài 4 (ba cây) và Bài 5 (đếm cặp tối ưu) khớp brute.')

Bài 4 (ba cây) và Bài 5 (đếm cặp tối ưu) khớp brute.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def diam(edges):
    n = len(edges) + 1; g = adjacency(edges)
    a = farthest(g, 0); return dist_farthest(g, a)
d1, d2 = diam(edges1), diam(edges2)
return max(d1, d2, (d1 + 1) // 2 + (d2 + 1) // 2 + 1)
```

**Checklist:**

- [ ] `n = len(edges) + 1` (cây một đỉnh có `edges` rỗng)?
- [ ] BFS **hai** lần (không lấy `ecc(0)` làm đường kính)?
- [ ] Làm tròn **lên** `⌈d/2⌉ = (d + 1) // 2`?
- [ ] Lấy `max` với `d1`, `d2`?
- [ ] Không dùng đệ quy sâu (cây đường thẳng `10^5`)?